In [ ]:
import os

name = "img000_RT000.png"
base_path = "/Volumes/T7/SPADES_frames"
representations = ["lnes", "three_c_representation", "to_voxel_grid", "two_d_histogram", "two_polarity_time_surface"]

traj = name.split("_")[1][:-4]

img_paths = []
# Collect images from different representations
for rep in representations:
    path = os.path.join(base_path, rep, traj, name)
    img_paths.append(path)


import os
import math
import matplotlib.pyplot as plt

rows, cols = 3, 2
slots = rows * cols

# 16:9 images (1280x720)
aspect = 1280 / 720  # 1.777...

# Choose how big each image cell should be (inches)
per_ax_width = 4.0            # tweak to taste
per_ax_height = per_ax_width / aspect

# Equal padding between rows/cols (inches)
pad_in = 0.05           # tweak to taste

# Figure size computed to fit the grid + pads
fig_w = cols * per_ax_width + (cols - 2) * pad_in
fig_h = rows * per_ax_height + (rows - 1) * pad_in

fig, ax = plt.subplots(rows, cols, figsize=(fig_w, fig_h),
                       constrained_layout=True, squeeze=False)
# Make row/col spacing equal (in inches)
fig.set_constrained_layout_pads(w_pad=pad_in, h_pad=pad_in, wspace=0, hspace=0)

# Fill left-to-right across each row
for i, img_path in enumerate(img_paths[:slots]):
    r, c = divmod(i, cols)
    ax[r, c].axis('off')
    if os.path.exists(img_path):
        img = plt.imread(img_path)
        ax[r, c].imshow(img, aspect='equal')  # preserves pixel aspect
        # bottom-left label in neon/cyan, bold
        label = chr(97 + i)
        ax[r, c].text(0.05, 0.05, label, fontsize=20, color='cyan',
                      fontweight='bold', transform=ax[r, c].transAxes)

# Hide any unused axes (if fewer than 6 images)
for j in range(len(img_paths[:slots]), slots):
    r, c = divmod(j, cols)
    ax[r, c].axis('off')

plt.show()

In [ ]:
for i, name in enumerate(representations):
    label = chr(97 + i)  # 'a' is 97 in ASCII
    print(f"{label}: {name}")

In [ ]:
df_metrics.head()

In [ ]:
df_metrics.sort_values(by="speed_score", ascending=True, inplace=True)
best = np.array(df_metrics["filename"][:3])
df_metrics.sort_values(by="speed_score", ascending=False, inplace=True)
worst = np.array(df_metrics["filename"][:3])
total = np.concatenate((best, worst))
print(total)
trajs = [int(name.split("_")[1][2:-4]) for name in total]
print(trajs)

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from scipy.spatial.transform import Rotation

def project_points(q, r, K):
    points = np.float32([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]]).reshape(-1, 3)

    rotV = np.expand_dims(Rotation.from_quat(q).as_rotvec(), axis=1)
    image_points, _ = cv2.projectPoints(points, rotV, r, K, distCoeffs=np.zeros(5))

    # Draw the axes on the image
    origin = tuple(map(int, image_points[0].ravel()))
    x_axis = tuple(map(int, image_points[1].ravel()))
    y_axis = tuple(map(int, image_points[2].ravel()))
    z_axis = tuple(map(int, image_points[3].ravel()))
    return origin, x_axis, y_axis, z_axis

import json
with open("predictions/camera.json", "r") as f:
    data = json.load(f)
    K = np.array(data["cameraMatrix"])
    print(K.dtype)

    
def to_float_array(x, expected_len=None):
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]
    arr = np.fromstring(s.replace(",", " "), sep=" ", dtype=np.float32)
    if expected_len is not None and arr.size != expected_len:
        raise ValueError(f"Expected {expected_len} floats, got {arr.size}: {x}")
    return arr

In [ ]:
df_keypoints = pd.read_json("predictions/kpts_predictions.json", lines=True)


df_predictions = df_metrics[df_metrics["filename"].isin(total)]
base_path = "/Volumes/T7/SPADES_frames/three_c_representation/"


rows, cols = 3, 2
slots = rows * cols

# 16:9 images (1280x720)
aspect = 1280 / 720  # 1.777...

# Choose how big each image cell should be (inches)
per_ax_width = 4.0            # tweak to taste
per_ax_height = per_ax_width / aspect

# Equal padding between rows/cols (inches)
pad_in = 0.05           # tweak to taste

# Figure size computed to fit the grid + pads
fig_w = cols * per_ax_width + (cols - 2) * pad_in
fig_h = rows * per_ax_height + (rows - 1) * pad_in

fig, ax = plt.subplots(rows, cols, figsize=(fig_w, fig_h),
                       constrained_layout=True, squeeze=False)

fig.set_constrained_layout_pads(w_pad=pad_in, h_pad=pad_in, wspace=0, hspace=0)

for i, filename in enumerate(df_predictions["filename"][:rows * cols]):
    r, c = divmod(i, cols)          # r = i // 2, c = i % 2
    ax[r, c].axis('off')

    traj = os.path.splitext(filename.split("_")[1])[0]
    img_path = os.path.join(base_path, traj, filename)
    row = df_predictions.loc[df_predictions["filename"] == filename].iloc[0]

    r_xyz = to_float_array(row["r_xyz"], expected_len=3)
    q_xyzw = to_float_array(row["q_xyzw"], expected_len=4)

    # kpts = df_keypoints[df_keypoints["filename"] == filename]
    # kpts = to_float_array(kpts["keypoints"], expected_len=16)
    # print(kpts)

    if os.path.exists(img_path):
        # --- Read with OpenCV (BGR, uint8) so original colors are preserved ---
        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            continue

        # project to pixel coords (ensure integer tuples for cv2)
        origin, x_axis, y_axis, z_axis = project_points(q_xyzw, r_xyz, K)
        origin = tuple(np.round(origin).astype(int))
        x_axis = tuple(np.round(x_axis).astype(int))
        y_axis = tuple(np.round(y_axis).astype(int))
        z_axis = tuple(np.round(z_axis).astype(int))

        # Neon BGR colors (cv2 is BGR!)
        PINK = (255,  64, 255)  # X-axis: neon magenta
        LIME = ( 40, 255, 120)  # Y-axis: neon green
        CYAN = (255, 255,   0)  # Z-axis: cyan

        # Draw with anti-aliased lines
        cv2.line(img_bgr, origin, x_axis, PINK, 4, cv2.LINE_AA)
        cv2.line(img_bgr, origin, y_axis, LIME, 4, cv2.LINE_AA)
        cv2.line(img_bgr, origin, z_axis, CYAN, 4, cv2.LINE_AA)

        # Convert once for Matplotlib display
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax[r, c].imshow(img_rgb)
        label = chr(97 + i)
        ax[r, c].text(0.05, 0.05, label, fontsize=20, color='cyan',
                      fontweight='bold', transform=ax[r, c].transAxes)

# Hide any leftover axes if fewer than 6 images
for j in range(i + 1, rows * cols):
    r, c = divmod(j, cols)
    ax[r, c].axis('off')

# plt.tight_layout()
plt.show()
    

In [ ]:
for i, name in enumerate(df_predictions["filename"]):
    label = chr(97 + i)  # 'a' is 97 in ASCII
    print(f"{label}: {name}")

In [ ]:
df_predictions